# `mboostrs` — distributional boosting (GAMLSS)

Standard regression boosting models a predictor for the conditional mean $E[y \mid x]$. Distributional boosting models predictors for multiple parameters of the conditional response distribution, allowing its spread or shape to vary with covariates.

**GAMLSS** (Generalised Additive Models for Location, Scale, and Shape) models *all parameters* of the conditional distribution simultaneously:

- **Location** ($\mu$, often the mean) — centre of the distribution
- **Scale** ($\sigma$) — spread of the distribution
- **Shape** ($\nu$, $\tau$) — skewness and kurtosis (for richer families)

Distributional models are useful for:
- **Heteroscedastic data** where variance depends on predictors
- **Bounded responses** (proportions, percentages)
- **Heavy-tailed data** where tail behaviour matters
- **Positive-skewed data** with a non-constant coefficient of variation

`mboostrs` provides `gamboostLSS` and `glmboostLSS` — the boosting equivalent of GAMLSS — matching R's `gamboostLSS` 2.2.0.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import altair as alt
import mboostrs as mb

rng = np.random.default_rng(42)
print("mboostrs version:", mb.__version__)

## 1. GaussianLSS — joint mean + variance (location-scale model)

A location-scale model has predictors for both the mean $\mu$ and log-standard-deviation $\log(\sigma)$.

Here the mean is linear in $x$ ($\mu = 1.5x$), while log-standard-deviation varies with $x$ ($\log\sigma = 0.3 + 0.4|x|$). A mean-only model does not represent this change in spread.

In [ ]:
n = 400
x = rng.uniform(-2, 2, n)
mu = 1.5 * x
log_sigma = 0.3 + 0.4 * np.abs(x)
y = rng.normal(mu, np.exp(log_sigma))
data = pl.DataFrame({"x": x, "y": y})

m_lss = mb.gamboostLSS(
    "y ~ bbs(x, df=4)",
    data=data,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=300, nu=0.1),
)
print(f"Parameters: {m_lss.param_names}")
print(f"Risk shape: {m_lss.risk_.shape} (iterations \u00d7 parameters)")

# Predict per parameter
mu_hat = m_lss.predict(parameter="mu", type="response")
sigma_hat = np.exp(m_lss.predict(parameter="sigma", type="link"))
print(f"mu predicted: mean={mu_hat.mean():.3f}")
print(f"sigma predicted: mean={sigma_hat.mean():.3f}")

In [ ]:
# Risk paths per parameter (GaussianLSS)
risk = m_lss.risk_
param_keys = list(risk.keys()) if isinstance(risk, dict) else ["mu", "sigma"]
risk_data = []
if isinstance(risk, dict):
    for param in risk:
        for m, r in enumerate(risk[param]):
            risk_data.append({"iteration": m, "risk": r, "parameter": param})
else:
    for m, r in enumerate(risk):
        risk_data.append({"iteration": m, "risk": r, "parameter": "total"})
risk_df = pl.DataFrame(risk_data)

alt.Chart(risk_df).mark_line().encode(
    x="iteration:Q", y="risk:Q", color="parameter:N",
    strokeDash="parameter:N",
).properties(title="Risk path per parameter \u2014 GaussianLSS")


## 2. Per-parameter formulas (asymmetric specification)

Each distributional parameter can have its own formula. Here, $\mu$ depends on both $x_1$ and $x_2$, while $\log(\sigma)$ depends only on $x_1$.

Pass a dictionary to `formula`: `{"mu": "...", "sigma": "..."}`.

In [ ]:
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
mu = 0.5 * x1 - 0.3 * x2
log_sigma = 0.2 + 0.1 * x1
y = rng.normal(mu, np.exp(log_sigma))
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y})

m_asy = mb.gamboostLSS(
    formula={"mu": "y ~ bbs(x1, df=4)",
             "sigma": "y ~ bbs(x1, df=3)"},
    data=data,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
for param in m_asy.param_names:
    pred = m_asy.predict(parameter=param, type="link")
    print(f"{param}: predicted mean={pred.mean():.3f}")

## 3. GammaLSS — positive response

For **positive-continuous** data, `GammaLSS` models both the mean ($\mu$) and the log-shape parameter ($\log(\nu)$). The Gamma distribution has a constant coefficient of variation when $\nu$ is fixed, but $\nu$ can also depend on predictors.

In [ ]:
n = 400
x = rng.uniform(0.5, 2, n)
mu_gamma = 0.5 + 1.5 * x
shape = 3.0
scale_gamma = mu_gamma / shape
y_gamma = rng.gamma(shape, scale_gamma)
data = pl.DataFrame({"x": x, "y": y_gamma})

m_gamma_lss = mb.gamboostLSS(
    "y ~ bbs(x, df=4)",
    data=data,
    families=mb.GammaLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"GammaLSS parameters: {m_gamma_lss.param_names}")

## 4. StudentTLSS — heavy-tailed response

The **Student t** distribution has an additional degrees-of-freedom parameter $\text{df}$ that controls tail heaviness. `StudentTLSS` models $\mu$, $\log(\sigma)$, and $\log(\text{df})$.

Use this family when a Gaussian model's tails are too light to represent occasional extreme observations.

In [ ]:
n = 500
x = rng.uniform(-2, 2, n)
mu_t = 0.8 * x
sigma_t = 0.5 + 0.2 * np.abs(x)
df_t = 4
y_t = mu_t + sigma_t * rng.standard_t(df_t, size=n)
data = pl.DataFrame({"x": x, "y": y_t})

m_t = mb.gamboostLSS(
    "y ~ bbs(x, df=4)",
    data=data,
    families=mb.StudentTLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"StudentT parameters: {m_t.param_names}")
mu_t_hat = m_t.predict(parameter="mu", type="response")
print(f"mu range: [{mu_t_hat.min():.2f}, {mu_t_hat.max():.2f}]")

In [ ]:
# StudentTLSS: fitted mu +/- 2 sigma band
mu_t_hat = m_t.predict(parameter="mu", type="response")
sigma_t_hat = np.exp(m_t.predict(parameter="sigma", type="link"))
sort_idx = np.argsort(x)
band_df = pl.DataFrame({
    "x": x[sort_idx],
    "mu": mu_t_hat[sort_idx],
    "upper": mu_t_hat[sort_idx] + 2 * sigma_t_hat[sort_idx],
    "lower": mu_t_hat[sort_idx] - 2 * sigma_t_hat[sort_idx],
})
raw_df = pl.DataFrame({"x": x, "y": y_t})

alt.layer(
    alt.Chart(band_df).mark_area(opacity=0.2, color="blue").encode(
        x="x", y="upper:Q", y2="lower:Q"
    ),
    alt.Chart(band_df).mark_line(color="blue").encode(
        x="x", y="mu:Q"
    ),
    alt.Chart(raw_df).mark_point(opacity=0.2, size=10).encode(
        x="x", y="y"
    ),
).properties(title="StudentTLSS \u2014 fitted mean \u00b1 2\u03c3")


## 5. BetaLSS — bounded (0, 1) response

The **Beta** distribution models continuous data on $(0, 1)$ — proportions, rates, percentages, concentration indices. `BetaLSS` models both $\mu$ (mean) and $\phi$ (precision).

The Beta distribution accounts for the bounded support; its variance depends on both the mean and precision.

In [ ]:
n = 300
x = rng.uniform(0, 1, n)
mu_beta = 0.2 + 0.6 * x
phi_beta = 5 + 3 * x
a = mu_beta * phi_beta
b = (1 - mu_beta) * phi_beta
y_beta = rng.beta(a, b)
data = pl.DataFrame({"x": x, "y": y_beta})

m_beta_lss = mb.gamboostLSS(
    "y ~ bbs(x, df=4)",
    data=data,
    families=mb.BetaLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"BetaLSS parameters: {m_beta_lss.param_names}")
mu_beta_hat = m_beta_lss.predict(parameter="mu", type="response")
print(f"mu range: [{mu_beta_hat.min():.3f}, {mu_beta_hat.max():.3f}]")

## 6. Per-parameter mstop

Different distributional parameters may require different amounts of boosting. You can set per-parameter `mstop` values — here we give the mean 200 iterations and the variance only 100.

In [ ]:
m_lss_pp = mb.gamboostLSS(
    "y ~ bbs(x, df=4)",
    data=data,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=300, nu=0.1),
    mstop={"mu": 200, "sigma": 100},
)
print(f"mstop: {m_lss_pp.mstop}")
print(f"Risk: {m_lss_pp.risk_[-1]:.4f}")

## 7. LSS Cross-validation

Cross-validation for GAMLSS models is more complex because we need to tune a grid of `mstop` values for each parameter. `cvrisk_glmboostLSS` and `cvrisk_gamboostLSS` handle this.

In [ ]:
n = 300
x = rng.uniform(-2, 2, n)
mu = 0.8 * x
log_sigma = 0.2 + 0.3 * np.abs(x)
y_cv = rng.normal(mu, np.exp(log_sigma))
data_cv = pl.DataFrame({"x": x, "y": y_cv})

# Pre-compute folds for reproducible CV
fold_matrix = mb.cv_weights(n, type="subsampling", B=5, prob=0.7)
grid = [[m, m] for m in range(0, 201, 10)]

# Linear LSS CV
cv_lin = mb.cvrisk_glmboostLSS(
    "y ~ x", data=data_cv,
    families=mb.GaussianLSS(),
    control=mb.boost_control(200, 0.1),
    folds=fold_matrix, grid=grid,
)
print(f"Linear LSS CV: best mstop per parameter = {cv_lin['mstop']}")

# GAM LSS CV (nonlinear)
cv_gam_lss = mb.cvrisk_gamboostLSS(
    "y ~ bbs(x, df=4)", data=data_cv,
    families=mb.GaussianLSS(),
    control=mb.boost_control(200, 0.1),
    folds=fold_matrix, grid=grid,
)
print(f"GAM LSS CV: best mstop per parameter = {cv_gam_lss['mstop']}")

## 8. Cyclic vs non-cyclic fitting

The fitting method determines how parameter-specific boosting updates are scheduled:

- **`method='cyclic'`** (default): update every parameter in turn at each global iteration.
- **`method='noncyclic'`**: update the parameter with the greatest improvement in overall risk at each global iteration. Each iteration updates one parameter.

The non-cyclic method can require fewer learner fits per global iteration; its risk path and parameter-specific stopping counts differ from the cyclic method.

In [ ]:
n = 300
x = rng.uniform(-2, 2, n)
mu = 1.0 * x
log_sigma = 0.3 + 0.2 * np.abs(x)
y_nc = rng.normal(mu, np.exp(log_sigma))
data_nc = pl.DataFrame({"x": x, "y": y_nc})

# Cyclic (default): all params updated each iteration
m_cyc = mb.glmboostLSS(
    "y ~ x", data=data_nc,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
    method="cyclic",
)
print(f"Cyclic risk: {m_cyc.risk_[-1]:.4f}")

# Non-cyclic: only best-improving param updated each iteration
m_ncyc = mb.glmboostLSS(
    "y ~ x", data=data_nc,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
    method="noncyclic",
)
print(f"Non-cyclic risk: {m_ncyc.risk_[-1]:.4f}")

In [ ]:
# Cyclic vs non-cyclic risk paths
def get_lss_risk(model):
    risk = model.risk_
    if isinstance(risk, dict):
        return np.sum(np.column_stack([risk[p] for p in risk]), axis=1)
    return risk

cyc_risk = get_lss_risk(m_cyc)
ncyc_risk = get_lss_risk(m_ncyc)
min_len = min(len(cyc_risk), len(ncyc_risk))
cmp_df = pl.DataFrame({
    "iteration": list(range(min_len)) * 2,
    "risk": np.concatenate([cyc_risk[:min_len], ncyc_risk[:min_len]]),
    "method": ["cyclic"] * min_len + ["non-cyclic"] * min_len,
})

alt.Chart(cmp_df).mark_line().encode(
    x="iteration:Q", y="risk:Q", color="method:N",
    strokeDash="method:N",
).properties(title="Cyclic vs non-cyclic fitting \u2014 total risk")


## 9. Real data: India childhood stunting

The **india** dataset contains childhood stunting measurements with demographic predictors. Stunting (height-for-age z-score) often exhibits heteroscedasticity — the variance changes with socioeconomic factors — which a location-scale model captures naturally.

In [ ]:
from mboostrs.datasets import india
india_df = india()
data_in = india_df
print(f"India data: {data_in.shape}")

m_india = mb.glmboostLSS(
    "stunting ~ bols(cbmi) + bols(cage) + bols(mbmi) + bols(mage) + bols(mcdist)",
    data=data_in,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"India parameters: {m_india.param_names}")
for param in m_india.param_names:
    c = m_india.coef(parameter=param)
    active = {k: v for k, v in c.items() if abs(v) > 0.001}
    print(f"{param} active coefficients: {active}")

## 10. LSS Stability Selection

Stability selection also works for GAMLSS models — identifying which predictors are consistently selected for each distributional parameter.

In [ ]:
ss_lss = mb.stabsel_glmboostLSS(
    "stunting ~ cbmi + cage + mbmi + mage + mcdist",
    data=data_in,
    families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
    q=3, PFER=1.0, B=30,
    sampling_type="SS", assumption="unimodal", seed=42,
)
if isinstance(ss_lss, dict) and "selected" in ss_lss:
    print(f"LSS stabsel: {len(ss_lss['selected'])} selected of {len(ss_lss['names'])}")
    print(f"Selected: {ss_lss['selected']}")
else:
    print(f"LSS stabsel returned type: {type(ss_lss)}")

## Available LSS families

In addition to the families demonstrated above, `mboostrs` provides: `NBinomialLSS`, `ZIPoLSS` (zero-inflated Poisson), `ZINBLSS` (zero-inflated negative binomial), `DirichletLSS` (compositional data), `LogNormalLSS`, `LogLogLSS`, and `WeibullLSS`.